**Curso: modelos de aprendizaje**

**UNIDAD 2**

---

*   NOMBRE: José Eduardo Martínez Fenelón
*   NO CONTROL: 23111506


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DIR = "/content/drive/MyDrive/CLASES/AGO-DIC 2025/MOEDLOS DE APRENDIZAJE/unidad 2/actividad RegresionLogistica"
# os.chdir(DIR) # Descomentar si se usa en Drive local

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías para preprocesamiento e ing de características
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder,StandardScaler, OrdinalEncoder,FunctionTransformer

# Librerías para la canalización
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline

# Librerías para la regresión
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_squared_error, r2_score, confusion_matrix, precision_score,recall_score,accuracy_score
from sklearn.decomposition import PCA

from sklearn.preprocessing import PolynomialFeatures

---
## **<font color=" #895cf9">Explica brevemente el uso de las librerias del segmento de codigo anterior, haciendo uso de la siguiente tabla</font>**

| Library | Explanation |
|----------|--------------|
| `import pandas as pd` | Manipulación y análisis de datos mediante DataFrames. |
| `import numpy as np` | Operaciones matemáticas y manejo de arreglos multidimensionales. |
| `import matplotlib.pyplot as plt` | Creación de visualizaciones estáticas, animadas e interactivas. |
| `import seaborn as sns` | Creación de gráficos estadísticos atractivos de alto nivel. |
| `sklearn.impute.SimpleImputer` | Imputación de valores nulos o faltantes en el dataset. |
| `sklearn.preprocessing.*` | Escalamiento (MinMax/Standard), codificación de variables y transformaciones. |
| `sklearn.compose.ColumnTransformer` | Aplicación de transformaciones específicas a distintas columnas. |
| `sklearn.pipeline.make_pipeline` | Creación de un flujo de trabajo de transformaciones y modelado. |
| `sklearn.linear_model.*` | Algoritmos predictivos lineales (Regresión Logística y Lineal). |
| `sklearn.metrics.*` | Evaluación del modelo (MSE, R2, matriz de confusión, recall, accuracy). |
| `sklearn.decomposition.PCA` | Reducción de dimensionalidad mediante Análisis de Componentes Principales. |
| `sklearn.preprocessing.PolynomialFeatures` | Generación de características polinomiales y de interacción. |

---

In [ ]:
#lectura del .csv con pandas
data_df = pd.read_csv('data.csv')
data_df.head()

# **Parte 1**. EDA

Haz que el `id` sea el índice del dataframe y efectúa una exploración inicial de los datos a través de:

In [ ]:
#utilice funcion .set_index()
#Guarde los cambios con inplace=True
data_df.set_index('id', inplace=True)
data_df.head()

1a) Estadísticas descriptivas para todas las variables del dataframe.

In [ ]:
#.describe para variables Numericas
data_df.describe()

In [ ]:
#.describe() para variable categoricas
data_df.describe(include=['O'])

1b) Valores únicos por variable para identificar posibles variables categóricas.

In [ ]:
# Se hace una lista de variables numéricas y otra de categóricas.
num_cols = data_df.select_dtypes(include=np.number).columns.tolist() #para variables numericas
cat_cols = data_df.select_dtypes(exclude=np.number).columns.tolist() #para variables Categoricas

In [ ]:
# Recuento de cada categoría única en las variables categóricas
for col in cat_cols:
    print(data_df[col].value_counts())
    print('-'*50)

1c) Búsqueda de valores faltantes.

In [ ]:
#Explica cada linea de codigo
print('Número valores faltantes: ') # Imprime el encabezado para el reporte en consola
for col in data_df.columns: # Itera cíclicamente sobre todas las columnas del DataFrame
  valor=sum(data_df[col].isna()) # Suma los valores nulos (NaN) encontrados en la columna iterada
  porcentaje=data_df[col].isna().mean()*100 # Calcula la proporción de nulos respecto al total de la muestra y la convierte en porcentaje
  print('{}:{}: porcentaje :{}'.format(col,valor,porcentaje)) # Formatea e imprime el nombre de la variable, cantidad de nulos y el %

1d) Diagrama de barras para determinar la frecuencia de los diagnósticos (cantidad de observaciones con resultado benigno y maligno)

In [ ]:
#utilice sns.countplot()
plt.figure(figsize=(6,4))
sns.countplot(data=data_df, x='diagnosis')
plt.title('Frecuencia de diagnósticos benignos (B) y malignos (M)')
plt.show()

2. Como hay tres valores relacionados con la misma característica (`mean`, `se` y `worst`) es muy probable que exista multicolinealidad en el conjunto.

La multicolinealidad en regresión es una condición que ocurre cuando algunas variables predictoras están fuertemente correlacionadas entre sí...

Para observar este efecto, elabora un mapa de calor que cuantifique la correlación de las variables numéricas en el dataframe.

In [ ]:
numeric_df = data_df.select_dtypes(include=[float, int])
plt.figure(figsize=(15, 12))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm')
plt.title('Mapa de calor de correlaciones')
plt.show()

Si te fijas en los valores de correlación entre las variables `_mean` y `_worst` es evidente la multicolinearidad.

Por ejemplo, la columna `radius_mean` tiene una correlación de 0.97 con la columna `radius_worst`.

Para solucionar el problema numérico de la multicolinealidad, tradicionalmente se recurre a eliminar variables. Conduciremos esta actividad en esos dos sentidos.

# **Parte 2**. Modelo con eliminación de variables altamente correlacionadas

Elimina las variables altamente correlacionadas:

3a) Ahora que sabes que las variables `_mean` y `_worst` tienen correlación alta, hay que quitar del dataframe un conjunto. Borra las columnas `_worst`.

In [ ]:
#para borrar las columnas _worst
worst_cols = [col for col in data_df.columns if '_worst' in col]
new_data_df = data_df.drop(columns=worst_cols)
new_data_df.head()

En las graficas de dispersion, los patrones lineales se refieren a la relacion entre dos variables que sigue una tendencia...

3b. Entre las variables `_mean`, identifica patrones lineales con diagramas de dispersión usando `sns.pairplot()`

In [ ]:
sns.pairplot(data=new_data_df[['radius_mean',
        'texture_mean',
        'perimeter_mean',
        'area_mean',
        'smoothness_mean',
        'compactness_mean',
        'concavity_mean',
        'concave points_mean',
        'symmetry_mean',
        'fractal_dimension_mean']])
plt.show()

De la matriz podrás observar relaciones lineales bastante evidentes entre:
* `radius_mean`, `perimeter_mean` y `area_mean`
* `compactness_mean`, `concavity_mean`, `concave_points_mean`

3c) Elabora otro mapa de calor confirmar con los valores de correlación.

In [ ]:
plt.figure(figsize=(15, 12))
sns.heatmap(new_data_df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Mapa de calor post-eliminación _worst')
plt.show()

3d) Después de observar los valores, nos quedaremos con sólo una variable de cada trío: `radius_mean` y `compactness_mean`. Elimina las restantes, no sólo del conjunto `_mean`, sino también de `_se`.

In [ ]:
#escriba TODAS las columnas que seran eliminadas
columnas_a_eliminar=['perimeter_mean', 'area_mean', 'concavity_mean', 'concave points_mean', 'perimeter_se', 'area_se', 'concavity_se', 'concave points_se']

#Convierte new_data_df en Dataframe usando .DataFrame() de la libreria de pandas
new_data_df = pd.DataFrame(new_data_df.drop(columns=columnas_a_eliminar))
new_data_df.head()

Observa la distribución de las variables resultantes (deben ser 12, sin contar la columna de indice):

4a) Utilizando histogramas. Guarda en una variable (`skew_cols`) las que tengan marcado sesgo positivo.

In [ ]:
#EXPLICA CADA LINEA DE CODIGO
skew_cols_df=pd.DataFrame() # Inicializa un DataFrame vacío para almacenar las columnas que presenten sesgo positivo
fig, axes =plt.subplots(4,3, figsize=(25,10)) # Crea una figura dividida en una matriz de subgráficos de 4 filas y 3 columnas
axes = axes.ravel() # Aplana el arreglo de ejes (2D a 1D) para facilitar la iteración a través de ellos
num_cols = new_data_df.select_dtypes(include=np.number).columns.tolist() # Obtiene la lista de nombres de las columnas que contienen datos numéricos

for col, ax in zip (num_cols, axes): # Inicia un bucle iterando en paralelo sobre cada columna y cada subgráfico disponible
  ax.hist(new_data_df[col]) # Dibuja un histograma con la distribución de los datos de la columna actual
  ax.set(title = f'{col}', xlabel=None) # Asigna como título del subgráfico el nombre de la variable evaluada
  if new_data_df[col].skew()>1 : skew_cols_df[col] =new_data_df[col] # Condición: Si el coeficiente de sesgo (skewness) supera 1, guarda la variable en skew_cols_df

skew_cols = skew_cols_df.columns.values # Extrae y guarda los nombres de las variables con asimetría en un arreglo denominado skew_cols

In [ ]:
skew_cols

4b) Dibujando box plots de todas las variables. Guarda en una variable (`scale_cols`) aquellas que no se encuentren en el intervalo [0,1]

In [ ]:
scale_cols_df=pd.DataFrame()
fig, axes = plt.subplots(4,3, figsize=(25,10))
axes = axes.ravel()

for col, ax in zip(new_data_df[num_cols], axes):
  ax.boxplot(new_data_df[col]) #COMPLETA LINEA DE CODIGO: Dibuja el gráfico de caja
  #COMPLETA EL IF: Verifica si los valores escapan del rango normalizado [0,1]
  if new_data_df[col].max() > 1 or new_data_df[col].min() < 0:
      scale_cols_df[col] = new_data_df[col]

scale_cols = scale_cols_df.columns.values

In [ ]:
scale_cols

Con todo el análisis anterior, estamos listos para generar un modelo *baseline* denominado `logr_model`. Para ello:

5a) Vuelve a leer el contenido del archivo, haz que el `id` sea el índice y separa las variables del dataframe...

In [ ]:
data_df = pd.read_csv('data.csv')
#AGREGA LINEA DE CODIGO PARA CONVERTIR COLUMNA "ID" EN INDICE
data_df.set_index('id', inplace=True)

In [ ]:
X = data_df.copy() #Se crea copia del df
# se elimina variable Categorica "diagnosis" de (X), debido a se sera utilizada como variable de salida (y)
X.drop('diagnosis', axis=1, inplace=True) #Caracteristica / variable Entrada
y = data_df['diagnosis'] #Objetivo /Variable salida

In [ ]:
#Libreria necesaria para realizar la particion del conjunto de datos
from sklearn.model_selection import train_test_split

# Se divide el conjunto en entrenamiento y prueba (80:20), es decir, 80% para Entrenamiento (train) y 20% para Prueba (test)
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.20, random_state=1)

5b) Prepara un transformador, denominado `preprocessing`, para borrar las columnas altamente correlacionadas...

In [ ]:
#EXPLIQUE CADA LINEA DE CODIGO
columnas_a_eliminar=['radius_worst','texture_worst','perimeter_worst','area_worst','smoothness_worst','compactness_worst','concavity_worst','concave points_worst','symmetry_worst','fractal_dimension_worst','perimeter_mean','area_mean','concavity_mean','concave points_mean','perimeter_se','area_se','concavity_se','concave points_se']
# Se listan explícitamente todas las columnas (las iteraciones _worst y redundancias detectadas _mean y _se) que serán descartadas del modelo.

preprocessing = ColumnTransformer([ # Inicializa un transformador de columnas que aplicará cambios estructurales previos al entrenamiento
    ('num', 'drop', columnas_a_eliminar)] # Define el paso 'num' ejecutando la acción 'drop' (eliminar) sobre las columnas listadas
   ,remainder='passthrough') # Garantiza que las variables restantes, no listadas, se mantengan intactas (passthrough)

5c) Entrena el modelo `logr_model` con el transformador `preprocessing` y regresión logística...

In [ ]:
#EXPLIQUE CADA LINEA DE CODIGO
logr_model=make_pipeline(preprocessing, LogisticRegression(max_iter=10000)) # Crea una tubería (pipeline) integrando el preprocesamiento seguido de la inicialización de la Regresión Logística
logr_model.fit(Xtrain,ytrain) # Entrena (ajusta) el modelo alimentando la tubería con el conjunto de datos y etiquetas de entrenamiento
predictions=logr_model.predict(Xtest) # Ejecuta el modelo entrenado con los datos de prueba (Xtest) para generar las predicciones

print('matriz de confusion 1:\n',confusion_matrix(ytest,predictions,labels=['B','M'])) # Compara y muestra los casos reales frente a los pronosticados (Verdaderos y Falsos - Positivos y Negativos)
print('recall    :\t',recall_score(ytest,predictions,pos_label='M')) # Imprime Sensibilidad (Recall): qué proporción de todos los tumores malignos reales el modelo detectó con éxito
print('precision :\t',precision_score(ytest,predictions,pos_label='M')) # Imprime Precisión: de las veces que el modelo predijo tumor maligno, cuántas veces era verdaderamente maligno
print('accurancy :\t',accuracy_score(ytest,predictions)) # Imprime Exactitud (Accuracy): evalúa la proporción total de predicciones (benignas y malignas) acertadas del modelo

<font color=" ORANGE">INTERPRETE LOS RESULTADOS OBTENIDOS EN MATRIZ DE CONFUSION Y METRICAS</font>

**INTERPRETACIÓN:**
El modelo baseline presenta un desempeño inicial bastante competente. La eliminación estratégica de columnas altamente correlacionadas (multicolinealidad) ayudó a sintetizar y fortalecer el modelo sin requerir exceso de variables. La matriz de confusión detalla el número exacto de aciertos y clasificaciones erróneas; las métricas de Recall y Precision superan ampliamente el 90%, indicando una potente capacidad de discriminación para detectar con precisión la presencia de malignidad sin un grado elevado de falsos positivos.

Para generar un modelo `logr_model2` con transformación y escalamiento:

6a) Modifica el transformador anterior para, además del borrado de las columnas correlacionadas, aplicar la raíz cuadrada...

In [ ]:
#EXPLIQUE CADA LINEA DE CODIGO
preprocessing2 = ColumnTransformer([ # Crea una segunda iteración del transformador de columnas, más complejo y robusto
    ('num0','drop', columnas_a_eliminar), # Etapa 1: Filtra el dataset eliminando características con alta colinealidad o redundancia
    ('num1',FunctionTransformer(np.sqrt),skew_cols), # Etapa 2: Mitiga el sesgo asimétrico de las variables en 'skew_cols' aplicando la función de raíz cuadrada
    ('num2',MinMaxScaler(),scale_cols) # Etapa 3: Estandariza la magnitud de las variables en 'scale_cols' acotándolas al rango uniforme [0, 1]
  ],remainder='passthrough') # Mantiene los datos de las columnas restantes que no cumplen estas condiciones sin alteraciones (passthrough)

6b) Entrena el modelo `logr_model2` con regresión logística, aplicando previamente el transformador modificado. Evalúa el modelo en el conjunto de prueba con las mismas métricas.

In [ ]:
#ESCRIBA CODIGO AQUI
logr_model2 = make_pipeline(preprocessing2, LogisticRegression(max_iter=10000))
logr_model2.fit(Xtrain, ytrain)
predictions2 = logr_model2.predict(Xtest)

print('matriz de confusion 2:\n', confusion_matrix(ytest, predictions2, labels=['B','M']))
print('recall    :\t', recall_score(ytest, predictions2, pos_label='M'))
print('precision :\t', precision_score(ytest, predictions2, pos_label='M'))
print('accurancy :\t', accuracy_score(ytest, predictions2))

<font color=" ORANGE">INTERPRETE LOS RESULTADOS OBTENIDOS EN MATRIZ DE CONFUSION Y METRICAS</font>

**INTERPRETACIÓN:**
La inclusión del `ColumnTransformer` para el escalamiento y la corrección del sesgo ha estabilizado sustancialmente los valores del conjunto de datos. En algoritmos como la regresión logística, esto permite optimizar los gradientes eficientemente e impide que las características con escalas numéricas absolutas enormes opaquen a otras. Se preserva y potencialmente mejora la generalización en el conjunto de pruebas, manteniendo métricas óptimas.

<font color=" ORANGE">CONCLUSION DE LA PRACTICA</font>

**CONCLUSIÓN:**
1. **Multicolinealidad:** Eliminar variables dependientes (como `_worst` y atributos `_mean` superfluos) mitiga la redundancia y optimiza el modelo predictivo previniendo el sobreajuste.
2. **Transformación y Escalamiento:** Controlar el sesgo de la distribución y limitar la escala de las características resulta elemental en el uso de algoritmos estocásticos y de gradiente. Esto otorga al modelo estabilidad numérica y asegura un aporte balanceado de variables en las clasificaciones finales.